# Pengambilan Data Spasial Kota Surabaya dari OpenStreetMap
Notebook ini mengambil data titik **pos pemadam kebakaran** dan **fasilitas kesehatan** (rumah sakit, puskesmas, klinik) di Kota Surabaya menggunakan Overpass API (OpenStreetMap).


In [7]:
import requests
import pandas as pd
import json
import time

OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
]

HEADERS = {
    "User-Agent": "TugasAkhir-DataCollection/1.0",
    "Accept": "application/json",
    "Content-Type": "application/x-www-form-urlencoded",
}

def query_overpass(query: str) -> dict:
    """Kirim query ke Overpass API via POST dengan headers."""
    for url in OVERPASS_URLS:
        try:
            print(f"Mencoba {url}...")
            resp = requests.post(
                url,
                data={"data": query},
                headers=HEADERS,
                timeout=120
            )
            resp.raise_for_status()
            print(f"Berhasil dari {url}")
            return resp.json()
        except requests.exceptions.RequestException as e:
            print(f"  Gagal: {e}")
            time.sleep(3)
            continue
    raise Exception("Semua server gagal. Tunggu 1-2 menit lalu coba lagi.")

def elements_to_df(elements: list) -> pd.DataFrame:
    """Konversi list elements Overpass ke DataFrame."""
    rows = []
    for el in elements:
        row = {"osm_id": el["id"], "type": el["type"]}
        if el["type"] == "node":
            row["lat"] = el["lat"]
            row["lon"] = el["lon"]
        elif "center" in el:
            row["lat"] = el["center"]["lat"]
            row["lon"] = el["center"]["lon"]
        tags = el.get("tags", {})
        row.update(tags)
        rows.append(row)
    return pd.DataFrame(rows)

print("Fungsi helper siap (3 server, headers lengkap).")

Fungsi helper siap (3 server, headers lengkap).


## 1. Pos Pemadam Kebakaran Kota Surabaya
Query OSM untuk tag `amenity=fire_station` di area Surabaya.


In [8]:
query_fire = """
[out:json][timeout:180];
area["name"="Surabaya"]["boundary"="administrative"]->.surabaya;
(
  node["amenity"="fire_station"](area.surabaya);
  way["amenity"="fire_station"](area.surabaya);
  relation["amenity"="fire_station"](area.surabaya);
);
out center body;
"""

result_fire = query_overpass(query_fire)
df_fire = elements_to_df(result_fire["elements"])

# Kolom yang relevan
cols_fire = ["osm_id", "type", "lat", "lon", "name", "addr:street", "addr:city"]
cols_fire = [c for c in cols_fire if c in df_fire.columns]
df_fire_clean = df_fire[cols_fire].copy()

print(f"Ditemukan {len(df_fire_clean)} pos pemadam kebakaran di Surabaya.")
df_fire_clean

Mencoba https://overpass-api.de/api/interpreter...
Berhasil dari https://overpass-api.de/api/interpreter
Ditemukan 12 pos pemadam kebakaran di Surabaya.


,osm_id,type,lat,lon,name,addr:city
0,449150188,way,-7.231831,112.785459,Pemadam Kebakaran Kecamatan Bulak,NaN
1,449809850,way,-7.295276,112.801729,Pos Pembantu Pemadam Kebakaran Kota Surabaya,NaN
2,451291149,way,-7.322084,112.713754,Dinas Kebakaran Pos Jambangan,NaN
3,451504869,way,-7.336707,112.754071,Dinas Kebakaran UPTD 3 Surabaya,NaN
4,451981973,way,-7.243063,112.759729,Dinas Kebakaran UPTD II Surabaya Utara Tambak ...,NaN
5,452387333,way,-7.303573,112.632906,Posko Pemadam Kebakaran,NaN
6,452582470,way,-7.314281,112.695842,Pemadam Kebakaran Surabaya IV,NaN
7,457996476,way,-7.340445,112.666022,Pos Pembantu Pemadam Kebakaran,NaN
8,459270806,way,-7.252277,112.653195,Dinas Pemadam Kebakaran Kandangan,NaN
9,460507505,way,-7.240293,112.625410,Dinas Pemadam Kebakaran UPTD Surabaya V Pos Pakal,NaN


## 2. Fasilitas Kesehatan Kota Surabaya
Query OSM untuk rumah sakit (`amenity=hospital`), puskesmas/klinik (`amenity=clinic`), dan dokter (`amenity=doctors`).


In [9]:
query_health = """
[out:json][timeout:60];
area["name"="Surabaya"]["boundary"="administrative"]->.surabaya;
(
  node["amenity"~"hospital|clinic|doctors"](area.surabaya);
  way["amenity"~"hospital|clinic|doctors"](area.surabaya);
  relation["amenity"~"hospital|clinic|doctors"](area.surabaya);
);
out center body;
"""

result_health = query_overpass(query_health)
df_health = elements_to_df(result_health["elements"])

cols_health = ["osm_id", "type", "lat", "lon", "name", "amenity", "addr:street", "healthcare"]
cols_health = [c for c in cols_health if c in df_health.columns]
df_health_clean = df_health[cols_health].copy()

print(f"Ditemukan {len(df_health_clean)} fasilitas kesehatan di Surabaya.")
print(f"  - hospital : {len(df_health_clean[df_health_clean['amenity']=='hospital'])}")
print(f"  - clinic   : {len(df_health_clean[df_health_clean['amenity']=='clinic'])}")
print(f"  - doctors  : {len(df_health_clean[df_health_clean['amenity']=='doctors'])}")
df_health_clean.head(10)


Mencoba https://overpass-api.de/api/interpreter...
Berhasil dari https://overpass-api.de/api/interpreter
Ditemukan 404 fasilitas kesehatan di Surabaya.
  - hospital : 71
  - clinic   : 319
  - doctors  : 14


,osm_id,type,lat,lon,name,amenity,addr:street,healthcare
0,4189210755,node,-7.322166,112.746204,Klinik Kencana Medika,clinic,Jl. Raya Kendangsari,clinic
1,4189218265,node,-7.288073,112.771706,Kliknik Spesial Meridian,clinic,Manyar Tirtomoyo 3,clinic
2,4191121402,node,-7.291510,112.753945,Scoliosis Correction Clinic,clinic,Jl. Ngagel Jaya Tengah,clinic
3,4191464390,node,-7.293404,112.735286,Medika Utama,clinic,Jl. Ciliwung,clinic
4,4191508335,node,-7.281890,112.736913,Miracle Aesthetic Clinic Thamrin,clinic,Jl. M H. Tamrin,clinic
5,4191509711,node,-7.280106,112.781764,Miracle Aesthetic Clinic Kertajaya Indah,clinic,Jl. Raya Kertajaya Indah Blok O,clinic
6,4192842880,node,-7.253353,112.748841,Dr. Silvester Hariwidodo D.Spa,doctors,Jl. Ketupa,doctor
7,4472384093,node,-7.257357,112.740929,Posyandu Genteng,clinic,NaN,clinic
8,4474589264,node,-7.250140,112.760249,Poliklinik Noor Mushola,clinic,NaN,clinic
9,4474589265,node,-7.252788,112.773696,Posyandu Ceria,clinic,NaN,NaN


## 3. Visualisasi pada Peta Interaktif
Tampilkan kedua dataset pada peta Leaflet menggunakan Folium.


In [10]:
import folium

# Pusat Surabaya
center_sby = [-7.2756, 112.7426]
m = folium.Map(location=center_sby, zoom_start=12, tiles="CartoDB positron")

# Layer pos pemadam kebakaran (merah)
fg_fire = folium.FeatureGroup(name="Pos Pemadam Kebakaran")
for _, row in df_fire_clean.iterrows():
    if pd.notna(row.get("lat")):
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=8,
            color="red",
            fill=True,
            fill_opacity=0.8,
            popup=f"<b>{row.get('name','N/A')}</b><br>OSM ID: {row['osm_id']}",
            tooltip=row.get("name", "Pos Damkar")
        ).add_to(fg_fire)
fg_fire.add_to(m)

# Layer fasilitas kesehatan (biru)
fg_health = folium.FeatureGroup(name="Fasilitas Kesehatan")
color_map = {"hospital": "blue", "clinic": "green", "doctors": "purple"}
for _, row in df_health_clean.iterrows():
    if pd.notna(row.get("lat")):
        c = color_map.get(row.get("amenity",""), "gray")
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=5,
            color=c,
            fill=True,
            fill_opacity=0.7,
            popup=f"<b>{row.get('name','N/A')}</b><br>Tipe: {row.get('amenity','')}<br>OSM ID: {row['osm_id']}",
            tooltip=row.get("name", row.get("amenity",""))
        ).add_to(fg_health)
fg_health.add_to(m)

folium.LayerControl().add_to(m)
print("Peta siap. Merah = Pos Damkar, Biru = RS, Hijau = Klinik, Ungu = Dokter")
m


Peta siap. Merah = Pos Damkar, Biru = RS, Hijau = Klinik, Ungu = Dokter


## 4. Ekspor ke CSV


In [11]:
df_fire_clean.to_csv("pos_pemadam_kebakaran_surabaya.csv", index=False)
df_health_clean.to_csv("fasilitas_kesehatan_surabaya.csv", index=False)

print("File tersimpan:")
print("  - pos_pemadam_kebakaran_surabaya.csv")
print("  - fasilitas_kesehatan_surabaya.csv")


File tersimpan:
  - pos_pemadam_kebakaran_surabaya.csv
  - fasilitas_kesehatan_surabaya.csv
